# 运算与类型转换

学习目标：能预测表达式的求值与转换，选择合适的相等判断和缺省值写法。

前置知识：原始类型、变量声明、对象属性、Node.js 脚本运行。

适用版本：ECMAScript 2025（ECMA-262 第 16 版）、Node.js 24.11.0；.mjs 文件按 ES 模块运行并采用严格模式。console 是宿主输出 API。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/03-operators-and-conversion/。

1. [main.mjs](scripts/03-operators-and-conversion/main.mjs)：按正文顺序运行全部正常示例。
2. [mixed-coalescing.mjs](scripts/03-operators-and-conversion/mixed-coalescing.mjs)：空值合并与逻辑或混写时，必须用括号说明分组。
3. [broken-chain.mjs](scripts/03-operators-and-conversion/broken-chain.mjs)：括号结束可选链后，继续读取 undefined 的属性会失败。

Step 1：从项目根目录进入本章工作目录。

```bash
cd content/编程语言/javascript
```

Step 2：运行全部正常示例，按各片段中的输出注释核对。

```bash
node scripts/03-operators-and-conversion/main.mjs
```

下文正常片段依次对应 main.mjs 中的代码；每段给出自身输入与定义。错误文件仅在相应小节单独运行。

## 1 算术与赋值

运算符把操作数连接成表达式。下表中的 x、y 表示本节参与运算的 Number 数值；本节的 x 初值为 7，y 为 2。

| 完整写法 | 中文名称／含义 |
| --- | --- |
| x + y | 加法；字符串参与时另有连接规则 |
| x - y | 减法 |
| x * y | 乘法 |
| x / y | 除法 |
| x % y | 余数 |
| x ** y | 乘方 |
| +x | 一元加，执行数值转换 |
| -x | 一元减，取相反数 |
| x++ | 后置递增，表达式取得修改前的值 |
| ++x | 前置递增，表达式取得修改后的值 |
| x-- | 后置递减 |
| --x | 前置递减 |

= 把右侧的值赋给左侧变量，并产生所赋的值。复合赋值如 +=、-=、*=、/=、%=、**= 将运算和回写合并；左侧目标只求值一次。余数的符号随被除数，不能直接当作数学中的非负模。

```javascript
const x = 7;
const y = 2;
console.log(x + y, x - y, x * y, x / y, x % y, x ** y);
console.log(+"7", -x, -7 % y);
let count = 3;
console.log(count++, ++count, count--, --count);
let total = 10;
total += 2;
total *= 3;
console.log(total);
// 输出依次为：
// 9 5 14 3.5 1 49
// 7 -7 -1
// 3 5 5 3
// 36
```

## 2 优先级与求值顺序

优先级决定表达式怎样分组，结合性解决同级运算的分组。乘除通常先于加减；乘方右结合；括号能明确改变分组。一元减直接出现在乘方左侧有语法限制，要明确写成对底数取负或对乘方结果取负。

分组不代表先执行右侧操作数。普通二元表达式从左到右求操作数的值，再按语法分组运算；短路运算会跳过某些操作数。下面用递增的取值顺序观察这个区别，实际业务代码应避免将多次修改压进一行。

```javascript
console.log(2 + 3 * 4, (2 + 3) * 4);
console.log(2 ** 3 ** 2, (-2) ** 2, -(2 ** 2));
let order = 1;
console.log(order++ + order++ * order++, order);
// 输出依次为：
// 14 20
// 512 4 -4
// 7 4
```

## 3 显式与隐式转换

Number()、String()、Boolean() 明确表达转换意图。Number() 将整段数值字符串转换为 Number；空字符串和只含空白的字符串转为 0，非数值文本转为 NaN。null 转为 0，undefined 转为 NaN。这些规则不是输入合法性校验。

二元加法先将对象转为原始值，再检查是否存在字符串；若有则连接，否则进行数值加法。减法等数值运算会转换操作数。普通对象向原始值转换可能调用 valueOf() 或 toString()，自定义对象还可能控制转换；不要把对象转换看作无副作用操作。这里先以原始值观察规则。Symbol 不能隐式转为字符串，但 String(symbol) 有显式转换的特殊支持。

```javascript
console.log(Number(" 12 "), Number(""), Number(null), Number(undefined));
console.log(Number("12px"), String(12), String(Symbol("course")));
console.log("12" + 3, "12" - 3, 1 + true);
console.log(Boolean("false"), Boolean(0));
// 输出依次为：
// 12 0 0 NaN
// NaN 12 Symbol(course)
// 123 9 2
// true false
```

## 4 真值与逻辑短路

条件判断使用布尔转换。在本章 Node 环境中，false、undefined、null、+0、-0、NaN、0n 和空字符串是假值；其余值是真值，包括空数组、空对象和字符串 "0"。

! 先转为布尔再取反。左侧是假值时，&& 直接返回左值；左侧是真值时，|| 直接返回左值。否则才计算并返回右值，所以它们不保证返回 Boolean。条件运算的写法是 condition ? whenTrue : whenFalse，三个名称分别表示条件、成立结果与不成立结果；只计算被选中的结果。

```javascript
console.log(Boolean([]), Boolean({}), Boolean(0n), Boolean(""));
let touches = 0;
const stopped = false && touches++;
const retained = "已有值" || touches++;
const chosen = touches === 0 ? "未访问右侧" : "已访问";
console.log(stopped, retained, touches, chosen);
console.log(!"", !!"0", 0 || 10, 3 && "继续");
// 输出依次为：
// true true false false
// false 已有值 0 未访问右侧
// true true 10 继续
```

## 5 空值合并与逻辑赋值

?? 只在左值为 null 或 undefined 时使用右值，适合保留有效的 0、false 或空字符串。它不能不加括号就与 && 或 || 混写。

??=、||=、&&= 分别在左侧为空值、假值、真值时求右值并回写，其他情况既不求右值，也不赋值。它们的判断条件与对应短路运算一致。

```javascript
console.log(0 ?? 10, false ?? true, "" ?? "默认");
console.log(null ?? "默认", (null ?? 0) || 5);
let quota = 0;
quota ??= 8;
let title = "";
title ||= "未命名";
let enabled = true;
enabled &&= false;
console.log(quota, title, enabled);
// 输出依次为：
// 0 false 
// 默认 5
// 0 未命名 false
```

## 6 可选链的保护范围

可选链用于访问可能不存在的对象。obj?.name 检查 obj 是否为空值，obj?.[key] 支持动态键，fn?.() 在函数值为空时跳过调用。这里 obj、key、fn 分别表示对象、属性键和预期可调用的值。

每个可能为空的层级都要放在正确的检查位置。连续链在空值处停止并得到 undefined，括号结束这段连续链；它不能保护未声明的根名称，也不能让存在但不可调用的值变成函数，不能作为赋值左侧。可选链只处理空值，不吞掉 getter 或函数内部抛出的异常。

```javascript
const profile = { contact: { city: "苏州" } };
const missing = null;
let keyReads = 0;
console.log(profile?.contact?.city, missing?.contact?.city);
console.log(missing?.[keyReads++], keyReads);
const plugin = {};
console.log(plugin.run?.() ?? "没有运行方法");
// 输出依次为：
// 苏州 undefined
// undefined 0
// 没有运行方法
```

## 7 比较、相等与特殊数值

<、<=、>、>= 用于关系比较。两个字符串按 UTF-16 码元序列比较，不按数值大小或汉语拼音排序；其他常见数值比较会转换操作数。范围判断要组合两次比较，不能照搬数学中的连续不等式。

=== 不进行跨类型转换；对象比较身份。== 按类型组合转换：布尔值会转成数值，数值与字符串比较会尝试数值转换，null 与 undefined 宽松相等。默认使用严格相等，确需宽松规则时明确输入范围。!== 和 != 分别是否定的严格与宽松相等。

Object.is() 是方法，不是运算符。它与 === 的关键数值差异是：将 NaN 视为同值，同时区分正零与负零。NaN 与任何值用 === 比较都不相等，和 NaN 的大小比较也不会成立。

```javascript
console.log("20" < "3", 20 < 3, 2 <= 2, 3 >= 4);
const score = 7;
console.log(0 <= score && score < 10);
console.log(2 === "2", 2 == "2", false == 0, null == undefined);
console.log(2 !== "2", 2 != "2");
console.log(NaN === NaN, Object.is(NaN, NaN), Number.isNaN(NaN));
console.log(0 === -0, Object.is(0, -0), 1 / 0, 1 / -0);
console.log(NaN < 1, NaN >= 1);
// 输出依次为：
// true false true false
// true
// false true true true
// true false
// false true true
// true false Infinity -Infinity
// false false
```

## 8 位运算与整数宽度

Number 的位运算先转换为 32 位整数，因此会截去小数并可能丢掉高位；不能用来处理任意大小的整数。下面 flags 是权限位，readBit 与 writeBit 各占一个二进制位。

& 按位与可检查位，| 按位或可设置位，^ 按位异或可翻转位，~ 按位取反。<< 左移，>> 保留符号右移，>>> 补零右移；Number 的移位量取低 5 位。它们对应的 &=、|=、^=、<<=、>>=、>>>= 复合赋值会回写结果。BigInt 的位运算不截成 32 位，但没有 >>>，数值运算不能混入 Number。

```javascript
const readBit = 0b001;
const writeBit = 0b010;
let flags = readBit | writeBit;
console.log(flags, (flags & readBit) !== 0);
flags ^= writeBit;
console.log(flags, ~0, 1 << 3, -8 >> 1, -8 >>> 1);
console.log(1 << 32, 4.9 | 0, (2 ** 32 + 1) | 0);
console.log(1n << 40n);
// 输出依次为：
// 3 true
// 1 -1 8 -4 2147483644
// 1 4 1
// 1099511627776n
```

## 9 独立观察错误边界

以下文件分别启动新进程；预期退出码为 1。先根据代码判断错误原因，再运行相应命令，核对错误名称及对应位置。错误消息全文由宿主决定。

空值合并与逻辑或混写时，必须用括号说明分组。

```javascript
console.log(null ?? false || true);
// 预期错误：SyntaxError；missing ) after argument list
```

Step 1：独立运行 scripts/03-operators-and-conversion/mixed-coalescing.mjs。

```bash
node scripts/03-operators-and-conversion/mixed-coalescing.mjs
```

括号结束可选链后，继续读取 undefined 的属性会失败。

```javascript
const user = null;
console.log((user?.profile).name);
// 预期错误：TypeError；Cannot read properties of undefined
```

Step 2：独立运行 scripts/03-operators-and-conversion/broken-chain.mjs。

```bash
node scripts/03-operators-and-conversion/broken-chain.mjs
```

## 本章小结

- 先看表达式分组，再看操作数顺序及是否短路；带副作用的表达式尤其要拆清楚。
- 明确转换和默认值策略：空值不等于假值，相等不等于关系比较。
- Number 位运算有 32 位边界；NaN 与正负零需要合适的判断方法。

## 练习

1. 为允许为 0 的库存设置缺省值 8，分别输入 0、null、undefined；核对结果应为 0、8、8。
2. 设计 3 个权限位，先设置其中两个，再清除一个；用按位与核对仅保留目标权限，不用十进制大小关系猜测。
3. 将 score 改为 -1、0、9、10，核对范围判断只接受 0、9；解释连续比较为什么不能替代两个判断。
4. 修复两个独立错误例，确认正常退出，并说明括号只是改变分组还是还需要补上 ?.。

### 提示

第 1 题比较 ?? 与 || 对 0 的处理；第 2 题清除某一位可与该位按位取反后的值做 &；第 4 题先确定连续可选链在哪里结束。

### 参考解析

库存用空值合并保留 0，null 与 undefined 才取 8。清除位时使用 flags &= ~目标位，核对相应按位与是否为零。范围判断分别比较下界与上界再用 && 组合；0 <= score < 10 会先产生布尔值，再把它与 10 比较。第一个错误可写为 (null ?? false) || true；第二个可写为 user?.profile?.name，保留空值时读取结果为 undefined。这些修复分别展示分组和连续可选链机制。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| TC39（tc39.es） | [§13.4–13.15 运算符、优先级语法与求值](https://tc39.es/ecma262/2025/multipage/ecmascript-language-expressions.html#sec-additive-operators)、[§7.1 类型转换](https://tc39.es/ecma262/2025/multipage/abstract-operations.html#sec-type-conversion)、[§7.2 相等及关系比较](https://tc39.es/ecma262/2025/multipage/abstract-operations.html#sec-testing-and-comparison-operations)、[§6.1.6 数值运算](https://tc39.es/ecma262/2025/multipage/ecmascript-data-types-and-values.html#sec-numeric-types)、[§20.1.2.15 Object.is](https://tc39.es/ecma262/2025/multipage/fundamental-objects.html#sec-object.is)、[§13.3.9 可选链](https://tc39.es/ecma262/2025/multipage/ecmascript-language-expressions.html#sec-optional-chains)、[§21.1.2.4 Number.isNaN](https://tc39.es/ecma262/2025/multipage/numbers-and-dates.html#sec-number.isnan)：ECMAScript 2025 规范规则与示例边界。 |